In [3]:
import pandas as pd
import numpy as np

In [5]:
file_path = "/mnt/Docs/Git_Repo/Credit_Scorecard/Data/interim/loan_data_binned.csv"

try:
    woe_df = pd.read_csv(file_path)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found. Please check the path and try again.")
except Exception as e:
    print(f"Error: An error occurred while loading the data: {e}")

woe_df.head()

Data loaded successfully.


,gender,education_level,home_ownership_status,loan_intent,previous_loan_defaults_on_file,loan_status,age_bin,income_binned,work_experience_binned,loan_income_ratio_bin,cr_hist_bin
0,female,Master,RENT,PERSONAL,No,1,young_adults,middle,entry_level,high,2-5
1,female,High School,OWN,EDUCATION,Yes,0,young_adults,low,entry_level,low,2-5
2,female,High School,MORTGAGE,MEDICAL,No,1,young_adults,low,entry_level,medium,2-5
3,female,Bachelor,RENT,MEDICAL,No,1,young_adults,middle,entry_level,medium,2-5
4,male,Master,RENT,MEDICAL,No,1,young_adults,middle,entry_level,high,2-5


In [7]:
woe_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44935 entries, 0 to 44934
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   gender                          44935 non-null  str  
 1   education_level                 44935 non-null  str  
 2   home_ownership_status           44935 non-null  str  
 3   loan_intent                     44935 non-null  str  
 4   previous_loan_defaults_on_file  44935 non-null  str  
 5   loan_status                     44935 non-null  int64
 6   age_bin                         44935 non-null  str  
 7   income_binned                   44935 non-null  str  
 8   work_experience_binned          44935 non-null  str  
 9   loan_income_ratio_bin           44935 non-null  str  
 10  cr_hist_bin                     44935 non-null  str  
dtypes: int64(1), str(10)
memory usage: 3.8 MB


In [8]:
def cal_woe_iv(df: pd.DataFrame, target: str) -> dict:
    """The function calculates the weight of evidence and information value for a given feature,
    where 1 represents a risk customer and 0 represents a no risk customer.
    Args: 
    df: pd.DataFrame - the input dataframe.
    target: str - the target variable (Risk_Flag).

    Rreturns: Two dictionaries holding WOE and IV values. 
    """
    non_event = 0
    event = 1
    total_no_good = (df[target] == non_event).sum()
    total_no_bad = (df[target] == event).sum()

    woe_dict_all = {}
    iv_dict = {}

    for col in df.drop(target,axis=1).columns:
        woe_dict_col = {}
        iv = 0

        grouped = df.groupby(col,observed=True)[target].value_counts().unstack(fill_value=0)

        for val in grouped.index:
            good = grouped.loc[val,non_event] if non_event in grouped.columns else 0
            bad = grouped.loc[val,event] if event in grouped.columns else 0

            # Add smoothing to avoid division by zero
            good = max(good, 0.50)
            bad = max(bad, 0.50)

            dist_good = good/total_no_good
            dist_bad = bad/total_no_bad

            woe = np.log(dist_good/dist_bad)
            woe_dict_col[val] = woe

            iv += (dist_good - dist_bad) * woe
        
        woe_dict_all[col] = woe_dict_col
        iv_dict[col] = iv
    
    return woe_dict_all,iv_dict

In [9]:
woe_dict, IV_dict = cal_woe_iv(woe_df,"loan_status")

In [10]:
IV_df = pd.DataFrame(list(IV_dict.items()), columns=["Variable","IV"])
IV_df["IV"] = round(IV_df["IV"],4)
IV_df = IV_df.sort_values(by="IV", ascending=False)

In [11]:
IV_df.loc[IV_df['IV'] <= 0.02]

,Variable,IV
7,work_experience_binned,0.0037
9,cr_hist_bin,0.0019
5,age_bin,0.0008
1,education_level,0.0003
0,gender,0.0000


*Generally, an __IV__ value less than __0.02__ indicates that the variable has little predictive power.
Here, none of the variables have an IV value less than 0.02, which suggests that all the variables have some predictive power in distinguishing between risky and non-risky customers. However, it's important to note that while IV can provide insights into the predictive power of individual variables, 
it should not be the sole criterion for variable selection. Other factors such as __multicollinearity__, __domain knowledge__, 
and __model performance__ should also be considered when selecting variables for a credit scorecard model.*


In [42]:
"""
Now, We will apply the logistic regression model from the statsmodels library to 
calculate the p-values for each feature. 
""" 

loan_data = pd.read_csv("loan_data_discretized.csv")

In [43]:
loan_data.columns

Index(['Age', 'Experience', 'Married/Single', 'House_Ownership',
       'Car_Ownership', 'Risk_Flag', 'Occupation_Risk_Profile',
       'State_Education_Rate', 'Income_lpa', 'income_binned', 'age_binned',
       'experience_binned'],
      dtype='str')

In [44]:
loan_data = loan_data.drop(columns=["income_binned","experience_binned","age_binned"])

In [45]:
numeric_cols = loan_data.drop(columns=["Risk_Flag"]).select_dtypes(include=[int,float]).columns.tolist()
categorical_cols = loan_data.drop(columns=["Risk_Flag"]).select_dtypes(exclude=[int,float]).columns.tolist()

In [49]:
loan_data = pd.get_dummies(loan_data, columns=categorical_cols, drop_first=True,dtype=int)

In [50]:
from sklearn.preprocessing import StandardScaler   
scaler = StandardScaler()
loan_data[numeric_cols] = scaler.fit_transform(loan_data[numeric_cols])

In [53]:
loan_data['Risk_Flag'] = loan_data['Risk_Flag'].map({"Not_Risky": 0, "Risky": 1})

In [55]:
import statsmodels.api as sm 
log_reg = sm.Logit(loan_data["Risk_Flag"], loan_data.drop(columns=["Risk_Flag"])).fit()

Optimization terminated successfully.
         Current function value: 0.378298
         Iterations 6


In [56]:
print(log_reg.summary())


                           Logit Regression Results                           
Dep. Variable:              Risk_Flag   No. Observations:               252000
Model:                          Logit   Df Residuals:                   251987
Method:                           MLE   Df Model:                           12
Date:                Sat, 04 Apr 2026   Pseudo R-squ.:                -0.01458
Time:                        19:43:41   Log-Likelihood:                -95331.
converged:                       True   LL-Null:                       -93961.
Covariance Type:            nonrobust   LLR p-value:                     1.000
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
Age                                    -0.0625      0.006    -10.439      0.000      -0.074      -0.051
Experience                             -0.0995      0.00